## Booking revenue baseline: `samples.wanderbricks.bookings`
Pre-run of Prompt 1a (no instructions, no skills, no MCP):
1. Table profile and data quality
2. Distribution of booking status
3. Monthly `total_amount` trend, split by status

Notes from the dry run:
* 72,247 bookings, `booking_id` is unique, every `property_id` joins to `properties` and every property to `destinations`.
* `created_at` runs from 2022-12-08 to 2025-07-30, so nothing after July 2025.
* **Cancelled** bookings: 15,285 rows, about 8.4M of `total_amount`. A baseline revenue number that doesn't filter `status = 'cancelled'` is overstated by roughly 21%. This is the 2d memory moment.
* One booking has `total_amount <= 0`.

In [ ]:
%sql
SELECT
  count(*)                                   AS total_rows,
  count(DISTINCT booking_id)                 AS distinct_booking_ids,
  count(DISTINCT property_id)                AS distinct_properties,
  count(DISTINCT user_id)                    AS distinct_users,
  count_if(total_amount IS NULL)             AS null_amounts,
  count_if(total_amount <= 0)                AS non_positive_amounts,
  count_if(check_out <= check_in)            AS invalid_stays,
  round(min(total_amount), 2)                AS min_amount,
  round(max(total_amount), 2)                AS max_amount,
  round(avg(total_amount), 2)                AS avg_amount,
  percentile_approx(total_amount, array(0.25, 0.5, 0.75)) AS amount_quartiles,
  round(avg(datediff(check_out, check_in)), 2) AS avg_nights,
  round(avg(guests_count), 2)                AS avg_guests,
  min(created_at)                            AS first_booking,
  max(created_at)                            AS last_booking
FROM samples.wanderbricks.bookings

In [ ]:
%sql
SELECT
  status,
  count(*)                                         AS bookings,
  round(100 * count(*) / sum(count(*)) OVER (), 2) AS pct_of_bookings,
  round(sum(total_amount), 2)                      AS total_amount,
  round(100 * sum(total_amount) / sum(sum(total_amount)) OVER (), 2) AS pct_of_amount
FROM samples.wanderbricks.bookings
GROUP BY ALL
ORDER BY bookings DESC

In [ ]:
%sql
SELECT
  date_trunc('month', created_at)                          AS month,
  count(*)                                                 AS bookings,
  round(sum(total_amount), 2)                              AS total_amount,
  round(sum(total_amount) FILTER (WHERE status <> 'cancelled'), 2) AS amount_excl_cancelled,
  round(sum(total_amount) FILTER (WHERE status = 'cancelled'), 2)  AS cancelled_amount
FROM samples.wanderbricks.bookings
GROUP BY ALL
ORDER BY month